## Analiza błędów i bias

### 📋 Cel notebooka
Ten notebook wykonuje szczegółową analizę błędów modelu BERT, w tym:
- **False Negatives** - krytyczne przypadki (pacjenci wymagający pilnej opieki błędnie sklasyfikowani)
- **Bias analysis** - analiza bias wg płci, wieku
- **Edge cases** - przypadki graniczne (niska pewność, blisko granicy decyzyjnej)
- **Systematic errors** - błędy systematyczne (które klasy są mylone)

### 🔄 Wymagania
- `bert_triage_model_*` - wytrenowany model (z notebooka 05)
- `corpus_val_*.txt` - **TYLKO zbiór walidacyjny** z notebooka 05

### ⚠️ WAŻNE: Data Leakage Prevention
- Używamy **TYLKO** zbioru walidacyjnego z notebooka 05
- **NIE** używamy całego corpus - to zapobiega nakładaniu się danych testowych z treningowymi

### 📊 Metodologia
- **False Negatives:** Przypadki, gdzie prawdziwa klasa to SOR/SPEC, a model przewidział POZ
- **Bias analysis:** Porównanie accuracy/precision/recall wg płci i wieku
- **Edge cases:** Przypadki z niską pewnością (< 0.6) lub blisko granicy decyzyjnej
- **Systematic errors:** Confusion matrix analysis - które klasy są mylone

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    precision_recall_fscore_support
)
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast
from tqdm.auto import tqdm
import re
from pathlib import Path
import glob
import sys

# Konfiguracja wizualizacji
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# ⚠️ AUTOMATYCZNE WYKRYWANIE NAJNOWSZYCH PLIKÓW
# Znajdź najnowszy model fine-tuned
triage_models = sorted([d for d in glob.glob("bert_triage_model_*") if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Nie znaleziono modelu bert_triage_model_*. Uruchom najpierw notebook 05_Fine_Tune_Model.ipynb")
MODEL_PATH = triage_models[0]
print(f"📂 Znaleziono model: {MODEL_PATH}")

# ⚠️ WAŻNE: Używamy TYLKO zbioru walidacyjnego z notebooka 05
val_split_files = sorted(glob.glob("corpus_val_*.txt"), reverse=True)
if not val_split_files:
    raise FileNotFoundError(
        "Nie znaleziono pliku corpus_val_*.txt.\n"
        "Uruchom najpierw notebook 05_Fine_Tune_Model.ipynb, który generuje train/val split."
    )
DATA_FILE = val_split_files[0]
print(f"📂 Używam zbioru walidacyjnego: {DATA_FILE}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

# Mapowanie klas (wspólny moduł — szukamy project root, gdzie jest label_mapping.py)
import os
current_dir = Path(os.getcwd())
project_root = None
if (current_dir / "label_mapping.py").exists():
    project_root = current_dir
elif (current_dir / "notebooks_ethos" / "label_mapping.py").exists():
    project_root = current_dir.parent
else:
    for parent in current_dir.parents:
        if (parent / "label_mapping.py").exists():
            project_root = parent
            break
if project_root is None:
    raise FileNotFoundError(
        f"Nie znaleziono pliku label_mapping.py. "
        f"Szukano w: {current_dir} i katalogach nadrzędnych."
    )
sys.path.insert(0, str(project_root))
print(f"📂 Dodano do sys.path: {project_root}")
from label_mapping import (
    LABEL_TO_ID,
    ID_TO_LABEL_SHORT,
    NUM_LABELS
)

print(f"🚀 Device: {DEVICE}")
print(f"📊 Liczba klas: {NUM_LABELS}")

In [ ]:
# 1. Ładowanie modelu i tokenizera
print("📥 Ładowanie modelu i tokenizera...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)
model.to(DEVICE)
model.eval()
print(f"✅ Model załadowany na {DEVICE}")

# 2. Wczytanie danych walidacyjnych
print("📂 Wczytywanie danych walidacyjnych...")
texts = []
labels = []
raw_sequences = []

with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split()
        if len(parts) < 2:
            continue
        
        label_str = parts[-1]
        input_text = " ".join(parts[:-1])
        
        if label_str in LABEL_TO_ID:
            texts.append(input_text)
            labels.append(LABEL_TO_ID[label_str])
            raw_sequences.append(line)

print(f"✅ Załadowano {len(texts)} próbek")
print(f"📊 Rozkład klas: {Counter([ID_TO_LABEL_SHORT[l] for l in labels])}")

In [ ]:
# 3. Ekstrakcja danych demograficznych
def extract_demographics(sequence):
    age = None
    gender = None
    age_match = re.search(r'AGE_(\d+)', sequence)
    if age_match:
        age = int(age_match.group(1))
    sex_match = re.search(r'(?:SEX|GENDER)_([MF])', sequence)
    if sex_match:
        gender = sex_match.group(1)
    return age, gender

def get_age_group(age):
    if age is None:
        return "Unknown"
    if age < 18:
        return "<18"
    elif age < 30:
        return "18-29"
    elif age < 50:
        return "30-49"
    elif age < 70:
        return "50-69"
    else:
        return "70+"

print("🔍 Ekstrakcja danych demograficznych...")
demographics = []
for seq in raw_sequences:
    age, gender = extract_demographics(seq)
    demographics.append({'age': age, 'gender': gender, 'age_group': get_age_group(age)})

df_demo = pd.DataFrame(demographics)
print(f"✅ Ekstrahowano dane dla {len(df_demo)} próbek")
print(f"📊 Rozkład płci: {df_demo['gender'].value_counts().to_dict()}")

In [ ]:
# 4. Wykonanie predykcji
class EthosEvalDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten()}

print("🔮 Wykonywanie predykcji...")
dataset = EthosEvalDataset(texts, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Predykcje"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        predictions = torch.argmax(logits, dim=-1)
        all_predictions.extend(predictions.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_predictions = np.array(all_predictions)
all_probs = np.array(all_probs)
all_labels = np.array(labels)

print(f"✅ Wykonano predykcje dla {len(all_predictions)} próbek")
print(f"📊 Accuracy: {(all_predictions == all_labels).mean():.3f}")

## 🔴 ANALIZA FALSE NEGATIVES (Krytyczne dla bezpieczeństwa)

False Negatives: prawdziwa klasa to SOR/SPEC, a model przewidział POZ

In [ ]:
# 5. Analiza False Negatives
URGENT_CLASSES = [1, 2, 3, 4]  # GASTROLOG, HEMATOLOG, NEFROLOG, SOR
POZ_CLASS = 0

false_negatives = []
for i in range(len(all_labels)):
    if all_labels[i] in URGENT_CLASSES and all_predictions[i] == POZ_CLASS:
        false_negatives.append({
            'index': i,
            'true_class': ID_TO_LABEL_SHORT[all_labels[i]],
            'pred_class': ID_TO_LABEL_SHORT[all_predictions[i]],
            'confidence': float(all_probs[i][all_predictions[i]]),
            'true_confidence': float(all_probs[i][all_labels[i]]),
            'sequence': raw_sequences[i],
            'age': df_demo.iloc[i]['age'],
            'gender': df_demo.iloc[i]['gender'],
            'age_group': df_demo.iloc[i]['age_group']
        })

df_fn = pd.DataFrame(false_negatives)
print(f"❌ Znaleziono {len(df_fn)} False Negatives")
if len(df_fn) > 0:
    print("\n📊 Rozkład wg prawdziwej klasy:")
    print(df_fn['true_class'].value_counts())
    print(f"\n📊 Średnia confidence (predykcja POZ): {df_fn['confidence'].mean():.3f}")

## ⚖️ ANALIZA BIAS (Płeć, Wiek)

In [ ]:
# 6. Analiza bias wg płci i wieku
bias_gender = {}
for gender in ['M', 'F']:
    mask = df_demo['gender'] == gender
    if mask.sum() == 0:
        continue
    gender_labels = all_labels[mask]
    gender_preds = all_predictions[mask]
    accuracy = (gender_preds == gender_labels).mean()
    precision, recall, f1, _ = precision_recall_fscore_support(gender_labels, gender_preds, average='weighted', zero_division=0)
    bias_gender[gender] = {'count': mask.sum(), 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

df_bias_gender = pd.DataFrame(bias_gender).T
print("⚖️ BIAS WG PŁCI:")
print(df_bias_gender)

bias_age = {}
for age_group in df_demo['age_group'].unique():
    if age_group == 'Unknown':
        continue
    mask = df_demo['age_group'] == age_group
    if mask.sum() == 0:
        continue
    age_labels = all_labels[mask]
    age_preds = all_predictions[mask]
    accuracy = (age_preds == age_labels).mean()
    precision, recall, f1, _ = precision_recall_fscore_support(age_labels, age_preds, average='weighted', zero_division=0)
    bias_age[age_group] = {'count': mask.sum(), 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

df_bias_age = pd.DataFrame(bias_age).T.sort_index()
print("\n⚖️ BIAS WG WIEKU:")
print(df_bias_age)

## 🔍 ANALIZA PRZYPADKÓW GRANICZNYCH (Edge Cases)

In [ ]:
# 7. Analiza edge cases
from scipy.stats import entropy

edge_cases = []
for i in range(len(all_labels)):
    probs = all_probs[i]
    max_prob = probs.max()
    sorted_probs = np.sort(probs)[::-1]
    prob_diff = sorted_probs[0] - sorted_probs[1] if len(sorted_probs) > 1 else 1.0
    entropy_val = entropy(probs + 1e-10)
    
    if max_prob < 0.6 or prob_diff < 0.1 or entropy_val > 1.5:
        edge_cases.append({
            'index': i,
            'true_class': ID_TO_LABEL_SHORT[all_labels[i]],
            'pred_class': ID_TO_LABEL_SHORT[all_predictions[i]],
            'is_correct': (all_predictions[i] == all_labels[i]),
            'max_confidence': float(max_prob),
            'prob_diff': float(prob_diff),
            'entropy': float(entropy_val)
        })

df_edge = pd.DataFrame(edge_cases)
print(f"🔍 Znaleziono {len(df_edge)} edge cases")
if len(df_edge) > 0:
    print(f"📊 Accuracy dla edge cases: {df_edge['is_correct'].mean():.3f}")

## 🔄 ANALIZA BŁĘDÓW SYSTEMATYCZNYCH

In [ ]:
# 8. Confusion Matrix i analiza błędów
cm = confusion_matrix(all_labels, all_predictions, labels=range(NUM_LABELS))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)],
            yticklabels=[ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)])
axes[0].set_title('Confusion Matrix (liczby)', fontweight='bold')
axes[0].set_xlabel('Predykcja')
axes[0].set_ylabel('Prawdziwa klasa')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=[ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)],
            yticklabels=[ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)])
axes[1].set_title('Confusion Matrix (procenty)', fontweight='bold')
axes[1].set_xlabel('Predykcja')
axes[1].set_ylabel('Prawdziwa klasa')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Najczęstsze błędy
error_pairs = defaultdict(int)
for i in range(len(all_labels)):
    if all_labels[i] != all_predictions[i]:
        true_class = ID_TO_LABEL_SHORT[all_labels[i]]
        pred_class = ID_TO_LABEL_SHORT[all_predictions[i]]
        error_pairs[(true_class, pred_class)] += 1

sorted_errors = sorted(error_pairs.items(), key=lambda x: x[1], reverse=True)
print("\n📊 Top 10 najczęstszych błędów:")
for (true_class, pred_class), count in sorted_errors[:10]:
    print(f"   {true_class} → {pred_class}: {count} przypadków")

In [ ]:
# 9. Raport podsumowujący
print("📋 RAPORT PODSUMOWUJĄCY\n")
print("=" * 60)
print(f"Całkowita liczba próbek: {len(all_labels)}")
print(f"Accuracy: {(all_predictions == all_labels).mean():.3f}")
print(f"\nFalse Negatives: {len(df_fn) if len(df_fn) > 0 else 0}")
print(f"Edge Cases: {len(df_edge) if len(df_edge) > 0 else 0}")
if len(sorted_errors) > 0:
    print(f"\nNajczęstszy błąd: {sorted_errors[0][0][0]} → {sorted_errors[0][0][1]} ({sorted_errors[0][1]} przypadków)")
print("=" * 60)